# AASIST - *Hugging Face*
## Evaluation & Benchmarking
Pre-trained model from [Hugging Face](https://huggingface.co/MTUCI/AASIST3)

### Objective
The primary objective of **AASIST3** is to provide a robust defense against speech deepfakes and synthetic audio attacks. It is designed specifically for the **ASVspoof 2024 Challenge**, aiming to distinguish between "bonafide" (human) speech and "spoof" (AI-generated or replayed) audio across diverse languages and recording conditions.

### Architecture Description
AASIST3 evolves the original AASIST (*Anti-spoofing with Adaptive Softmax and Instance-wise Temperature*) framework by integrating **Kolmogorov-Arnold Networks (KAN)** and Self-Supervised Learning (SSL) features.


The architecture is structured into the following functional stages:

1.  **SSL Feature Extraction:**
    The model utilizes a **Wav2Vec2** encoder as a front-end to extract high-level representations from raw audio waveforms. This allows the model to benefit from robust features learned during large-scale self-supervised pre-training.

2.  **KAN Bridge & Transformation:**
    Unlike traditional architectures that rely on standard MLP/Linear layers, AASIST3 incorporates **KAN Linear Layers**. These layers use learnable activation functions on the edges (splines) rather than fixed activations on nodes, allowing for more complex and efficient feature transformation.

3.  **Residual Encoding:**
    The extracted features pass through a series of **Residual Blocks** to capture hierarchical spectral and temporal patterns, ensuring stable gradient flow during training.

4.  **Graph Attention Networks (GAT):**
    To model the relationship between different segments of the audio signal, the architecture employs two specialized graph modules:
    * **GAT-S (Spatial):** Focuses on modeling dependencies across different frequency bins or feature dimensions.
    * **GAT-T (Temporal):** Focuses on modeling the long-term temporal dependencies of the speech signal.

5.  **Multi-branch Inference & Output:**
    The model uses four parallel inference branches integrated with **master tokens** to aggregate global information. The final classification (bonafide vs. spoof) is performed by an output layer also powered by KAN, which provides the final decision logic.

## Repo cloning and model import

In [2]:
!git clone https://github.com/mtuciru/AASIST3.git
!cd AASIST3

Cloning into 'AASIST3'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 69 (delta 22), reused 69 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 36.43 KiB | 4.05 MiB/s, done.
Resolving deltas: 100% (22/22), done.


### Dependencies installation

In [ ]:
#!pip install -r '/content/AASIST3/requirements.txt'

In [ ]:
# 1. Purge all potentially conflicting packages
!pip uninstall -y torch torchvision torchaudio torchcodec datasets

# 2. Install the strictly aligned PyTorch ecosystem (CUDA 12.1)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Install remaining dependencies, pinning datasets to the stable 2.x branch
!pip install transformers accelerate datasets==2.19.1 soundfile

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 92.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

> **RuntimeError:** Could not load libtorchcodec
This error is a binary linkage failure. It occurs when the Python wrapper for torchcodec cannot initialize its underlying C++ engine.

[Post adressing this issue](https://discuss.huggingface.co/t/issue-with-torchcodec-when-fine-tuning-whisper-asr-model/169315/2)

In [ ]:
# Colab VM or Linux
#!apt-get update && apt-get install -y ffmpeg
#!pip install -U "datasets[audio]" "torch==2.8.*" "torchcodec==0.7.*"
# HF docs: audio decoding uses TorchCodec + FFmpeg
# https://huggingface.co/docs/datasets/en/audio_load

# Evaluation

In [ ]:
import sys
import os

import torch
import torch.nn.functional as F
import torchaudio

import pandas as pd
import numpy as np

### Import pretrained model: *AASIST3*

In [ ]:
# Add the repository root to the search path
repo_root = "/content/AASIST3/"
if repo_root not in sys.path:
    sys.path.append(repo_root)

In [ ]:
# Disables the 'torchcodec' backend in torchaudio.
# Used to force the use of legacy backends (like ffmpeg or sox) or to avoid
# experimental decoder issues that might affect audio feature extraction consistency.
os.environ["TORCHAUDIO_USE_TORCHCODEC"] = "0"

In [ ]:
from model import aasist3

# Load the model from Hugging Face Hub
model = aasist3.from_pretrained("MTUCI/AASIST3")

In [ ]:
# Forces CUDA kernels to run synchronously.
# Essential for debugging; it ensures that GPU errors are reported at the
# exact line of Python code that triggered them, rather than asynchronously later.
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

### Dataset Preprocessing: *ASVspoof_2019_LA*

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

# 1. Setup Device and Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. Load Dataset (Streaming)
ds = load_dataset("Bisher/ASVspoof_2019_LA", split="test", streaming=True)

def preprocess_audio(audio_data, sr):
    # Convert numpy to torch tensor and ensure Float32
    audio = torch.from_numpy(audio_data).float().unsqueeze(0)

    # A. Resampling to 16kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        audio = resampler(audio)

    # B. Mono Check
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)

    # C. Pre-emphasis Filter: y[n] = x[n] - 0.97 * x[n-1]
    audio = torch.cat((audio[:, :1], audio[:, 1:] - 0.97 * audio[:, :-1]), dim=1)

    # D. Z-Score Normalization
    audio = (audio - audio.mean()) / (audio.std() + 1e-7)

    # E. Temporal Shaping (Exactly 64,600 samples)
    target_len = 64600
    current_len = audio.shape[1]

    if current_len < target_len:
        audio = F.pad(audio, (0, target_len - current_len))
    else:
        audio = audio[:, :target_len]

    return audio

# 3. Collection Loop
scores = []
labels = []
max_samples = 500  # Set to None to evaluate the entire test set

print(f"Starting evaluation on {device}...")

for i, sample in enumerate(tqdm(ds, desc="Processing samples")):
    if max_samples and i >= max_samples:
        break

    # Extract data
    audio_data = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Ground Truth Label: 0 for Bonafide, 1 for Spoof
    is_spoof = sample["key"] == "spoof"
    label = 0 if is_spoof else 1

    # Preprocess
    processed_audio = preprocess_audio(audio_data, sr).to(device)

    # Inference
    with torch.no_grad():
        output = model(processed_audio)
        # We extract the probability for the "Bonafide" class (Index 0)
        # Standard EER/t-DCF scripts expect higher scores for human speech
        probs = torch.softmax(output, dim=1)
        bonafide_score = probs[0][0].item()

    scores.append(bonafide_score)
    labels.append(label)

print(f"\nCollection complete. Collected {len(scores)} samples.")

Starting evaluation on cuda...


Processing samples: 0it [00:01, ?it/s]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Assuming scores and labels are populated lists
eval_results = pd.DataFrame({
    'score': scores,
    'label': labels
})

In [ ]:
from utils.metrics import compute_eer, compute_tDCF

bonafide_scores = scores[labels == 1]
spoof_scores = scores[labels == 0]

if bonafide_scores.size == 0 or spoof_scores.size == 0:
    print(f"Error: Missing class. Bonafide: {bonafide_scores.size}, Spoof: {spoof_scores.size}")
else:
    eer, frr, far, thresholds = compute_eer(bonafide_scores, spoof_scores)

print(f"EER: {eer * 100:.4f}%")
print(f"Optimal Threshold: {threshold}")

Error: Missing class. Bonafide: 500, Spoof: 0
EER: nan%
Optimal Threshold: 0.026405234053730964


In [ ]:
threshold = 0.6

# Assuming 1 = Bonafide, 0 = Spoof
# A prediction is Bonafide if score >= threshold
eval_results['prediction'] = (eval_results['score'] >= threshold).astype(int)

# False Positive (FP): Spoof (0) predicted as Bonafide (1)
eval_results['is_fp'] = (eval_results['label'] == 0) & (eval_results['prediction'] == 1)

# False Negative (FN): Bonafide (1) predicted as Spoof (0)
eval_results['is_fn'] = (eval_results['label'] == 1) & (eval_results['prediction'] == 0)

# General error flag
eval_results['is_error'] = eval_results['label'] != eval_results['prediction']

In [ ]:
eval_results.describe()

,score,label,prediction
count,500.000000,500.0,500.000000
mean,0.882275,1.0,0.882000
std,0.278874,0.0,0.322931
min,0.027405,1.0,0.000000
25%,0.980922,1.0,1.000000
50%,0.999999,1.0,1.000000
75%,1.000000,1.0,1.000000
max,1.000000,1.0,1.000000


---

### Dataset class

In [1]:
from torch.utils.data import Dataset, DataLoader

class ASVspoofDataset(Dataset):
    def __init__(self, hf_dataset, max_samples=None):
        # Converting streaming dataset to list for indexed access
        # (or use a wrapper that handles streaming)
        self.data = list(hf_dataset.take(max_samples)) if max_samples else list(hf_dataset)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        item = self.data[idx]
        audio_data = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]

        # Reuse your existing preprocess_audio function
        # Ensure it returns a tensor of shape [64600]
        feature = preprocess_audio(audio_data, sr).squeeze(0)

        # Ground Truth and Metadata
        utterance_id = item["audio_file_name"]
        label = item["key"]  # This is the integer 0 or 1

        # Order must be (Tensor, String, Tensor) to match: batch_x, utt_id, batch_y
        #return feature, utterance_id, torch.tensor(label)
        return feature, utterance_id, torch.tensor(label, dtype=torch.long)

You should download the official evaluation trial file from [Kaggle](https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset/data)
> Look into: ´LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt´

[Documentation](https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset/data)

In [ ]:
# Initialize Dataset and Loader
eval_ds = ASVspoofDataset(ds, max_samples=500)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, num_workers=2)

# Placeholders for paths
save_path = "results/scores.txt"
trial_path = "protocols/ASVspoof2019.LA.cm.eval.trl.txt" # Path to official protocol
loss_fn = torch.nn.CrossEntropyLoss()

### Dataset wrapper

In [ ]:
import torch.nn as nn

class ModelWrapper(nn.Module):
    def __init__(self, original_model):
        super(ModelWrapper, self).__init__()
        self.model = original_model

    def forward(self, x, **kwargs):
        # 1. Catch and ignore 'random' and 'dropout' via **kwargs
        out = self.model(x)

        # 2. Check if it's a tuple (features, logits) and unpack
        if isinstance(out, tuple):
            # In AASIST models, logits are typically the second element
            logits = out[1]
            features = out[0]
        else:
            logits = out
            features = torch.zeros((logits.size(0), 1), device=x.device)

        # 3. Handle Index Mismatch
        # The library's produce_evaluation_file uses batch_out[:, 1] for Bonafide.
        # As our model's Bonafide is at Index 0, we must swap them.
        # [Bonafide, Spoof] -> [Spoof, Bonafide]
        swapped_logits = logits[:, [1, 0]]

        return features, swapped_logits

In [ ]:
# Wrap your existing model
model_wr = ModelWrapper(model)
model_wr.to(device)
model_wr.eval()
print("Wrapping completed")

Wrapping completed


## Evaluation file

In [ ]:
from utils.metrics import produce_evaluation_file

# 1. Create a mini-protocol file from your 500 samples
mini_trial_path = "protocols/mini_eval_protocol.txt"

with open(mini_trial_path, "w") as f:
    # Iterate over the internal list of dictionaries in your dataset
    for item in eval_ds.data:
        utt_id = item["audio_file_name"]
        # ASVspoof protocol needs the string "bonafide" or "spoof"
        label_str = item["key"]

        # Format: [Speaker] [Utterance] [-] [SystemID] [Label]
        # We use dummy 'LA_0000' and '-' for compatibility
        f.write(f"LA_0000 {utt_id} - - {label_str}\n")

print(f"Mini protocol created with {len(eval_ds.data)} lines.")

# 2. Now run the execution
produce_evaluation_file(
    data_loader=eval_loader,
    model=model_wr,
    device=device,
    loss_fn=loss_fn,
    save_path="results/scores.txt",
    trial_path=mini_trial_path,
    max_batches=None
)

Mini protocol created with 500 lines.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from utils.metrics import produce_evaluation_file

# Execution
produce_evaluation_file(
    data_loader=eval_loader,
    model=model_wr,
    device=device,
    loss_fn=loss_fn,
    save_path=save_path,
    trial_path=trial_path,
    max_batches=None # Process all batches in the loader
)

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


---

## Alternative strategy 1: Running metrics functions

In [ ]:
from utils.metrics import compute_eer, compute_det_curve, compute_tDCF

In [ ]:
%%writefile .env
# Path to the ASVspoof 2019/2021/2024 dataset
DATASET_DIR=/path/to/your/dataset

# Path to the pre-trained weights or the model you want to validate
CHECKPOINT_PATH=./models/weights.pth

# Optional: WandB configuration for logging validation metrics
WANDB_API_KEY=your_api_key_here
WANDB_PROJECT=aasist3_validation

# Hardware configuration
CUDA_VISIBLE_DEVICES=0

Writing .env


In [ ]:
# Check if file was sucessfuly created
!ls -a | grep .env
!cat .env

.env
# Path to the ASVspoof 2019/2021/2024 dataset
DATASET_DIR=/path/to/your/dataset

# Path to the pre-trained weights or the model you want to validate
CHECKPOINT_PATH=./models/weights.pth

# Optional: WandB configuration for logging validation metrics
WANDB_API_KEY=your_api_key_here
WANDB_PROJECT=aasist3_validation

# Hardware configuration
CUDA_VISIBLE_DEVICES=0


In [ ]:
from torch.utils.data import DataLoader

# 3. Setup DataLoader
# Ensure your Dataset class handles the ASVspoof protocol format
eval_dataset = ASVspoofDataset(protocol_path="path/to/eval_protocol.txt")
data_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# 4. Define Loss Function and Paths
loss_fn = nn.CrossEntropyLoss()
save_path = "output_scores.txt"
trial_path = "path/to/eval_trials.txt"

In [ ]:
from utils.validation import produce_evaluation_file

produce_evaluation_file(
    data_loader=data_loader,
    model=model,
    device=device,
    loss_fn=loss_fn,
    save_path=save_path,
    trial_path=trial_path,
    random=False,
    dropout=0,
    max_batches=None # Set an integer if you want a partial run for debugging
)

In [ ]:
from utils.validation import compute_scores
compute_scores()

ValueError: too many values to unpack (expected 2)

---

## Alternative strategy 2: Importing metrics file

Download oficial evaluation metrics from AVSpoof2021 contest.

In [ ]:
!wget https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py

--2026-03-28 23:28:22--  https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18819 (18K) [text/plain]
Saving to: ‘eval_metrics.py’

eval_metrics.py     100%[===================>]  18.38K  --.-KB/s    in 0s      

2026-03-28 23:28:22 (187 MB/s) - ‘eval_metrics.py’ saved [18819/18819]



In [ ]:
# Load Protocol (cols: Speaker, Utterance, -, Attack, Label)
# Example file: ASVspoof2019.LA.cm.test.trl.txt
protocol_path = "/protocols/ASVspoof2019.LA.cm.test.trl.txt"
protocol = pd.read_csv(protocol_path, sep=" ", header=None,
                       names=['speaker', 'utt_id', 'dash', 'system', 'label'])

# Load your generated CM scores
cm_scores = pd.read_csv(cm_scores_file, sep=" ", header=None, names=['utt_id', 'score'])

# Merge to align Ground Truth with your Predictions
eval_data = pd.merge(protocol, cm_scores, on='utt_id')

# Split into Target (Bonafide) and Non-Target (Spoof) for EER calculation
target_scores = eval_data[eval_data['label'] == 'bonafide']['score'].values
nontarget_scores = eval_data[eval_data['label'] == 'spoof']['score'].values